# Reddit data NLP Toolbox

## Mandatory setup

This must be run for the rest of the notebook to work.

In [ ]:
#Install packages
# !pip install asent &> /dev/null
!python -m spacy download en_core_web_sm &> /dev/null

In [ ]:

#Load packages
import os, re, sqlite3, collections, spacy, pickle, csv, random
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from spacy import displacy
from spacy.matcher import Matcher
# import asent
from math import log10
from random import sample


#This prepares spacy for working on english
nlp = spacy.load('en_core_web_sm')
# add asent to pipeline
# nlp.add_pipe("asent_en_v1")
#This prepares a sentiment analyser
sentMachine = SentimentIntensityAnalyzer()
Vader_dictionary = sentMachine.make_lex_dict()

#!pip install contextualSpellCheck &> /dev/null
#import contextualSpellCheck #<- not used

#And this makes a spellchecker, also with spacy
#spellchecker = spacy.load('en_core_web_sm', disable=["tok2vec",'attribute_ruler','lemmatizer'])
#contextualSpellCheck.add_to_pipe(spellchecker)

#!pip install truecase &> /dev/null
#import truecase

#This function counts how many comments were in a list of subreddits
def count_comments(list_subreddits, keyword = "ALL"):
    if keyword == "ALL":        
        if list_subreddits[0] == "ALL":
            #Open connection
            connection = sqlite3.connect('/kaggle/input/reddit-comments-may-2015/database.sqlite')
            cursor = connection.cursor()
            #Get out texts from current subreddit
            cursor.execute("SELECT COUNT(*) FROM May2015")
            countresults = cursor.fetchall()
            print('in total: ' + str(countresults[0][0]) + ' comments')  
        
        else:
            for subreddit_name in list_subreddits:
                #Open connection
                connection = sqlite3.connect('/kaggle/input/reddit-comments-may-2015/database.sqlite')
                cursor = connection.cursor()
                #Get out texts from current subreddit
                cursor.execute("SELECT COUNT(*) FROM May2015 WHERE subreddit = '{}'".format(subreddit_name))
                countresults = cursor.fetchall()
                print(subreddit_name + ':  ' + str(countresults[0][0]) + ' comments')
    else:
        if list_subreddits[0] == "ALL":
            #Open connection
            connection = sqlite3.connect('/kaggle/input/reddit-comments-may-2015/database.sqlite')
            cursor = connection.cursor()
            #Get out texts from current subreddit
            cursor.execute("SELECT COUNT(*) FROM May2015 WHERE body LIKE '%{}%'".format(keyword))
            countresults = cursor.fetchall()
            print('in total: ' + str(countresults[0][0]) + ' comments')  
        else:
            for subreddit_name in list_subreddits:
                #Open connection
                connection = sqlite3.connect('/kaggle/input/reddit-comments-may-2015/database.sqlite')
                cursor = connection.cursor()
                #Get out texts from current subreddit
                cursor.execute("SELECT COUNT(*) FROM May2015 WHERE subreddit = '{}' AND body LIKE '%{}%'".format(
                    subreddit_name, keyword))
                countresults = cursor.fetchall()
                print(subreddit_name + ':  ' + str(countresults[0][0]) + ' comments')
    #Close connection
    connection.close()

#This function gets the comments out that is specified by the user
def get_specific_comments(list_subreddits, keyword = "ALL", number_comments = 2000):
    #Make empty dataframe
    out_df = pd.DataFrame(columns=['body'])
    #Go through each subreddit name
    for subreddit_name in list_subreddits:
        #Open connection
        connection = sqlite3.connect('/kaggle/input/reddit-comments-may-2015/database.sqlite')
        if keyword == 'ALL':
            #Get out comments from current subreddit
            out_df_small = pd.read_sql_query("SELECT body, subreddit, name FROM May2015 WHERE subreddit = '{}' LIMIT {}".format(subreddit_name, number_comments), connection)
        else:
            #Get out comments from current subreddit
            out_df_small = pd.read_sql_query("SELECT body, subreddit, name FROM May2015 WHERE subreddit = '{}' AND body LIKE '%{}%' LIMIT {}".format(subreddit_name, keyword, number_comments), connection)
        #Append the dataset to the large dataset
        out_df = out_df.append(out_df_small)
        #Print progress
        print("I got through " + subreddit_name)
        # Close connection
        connection.close()
    return out_df

#These two functions gets out a string in its context
def concordance_list(comment, searchword, width):
    token_list = [str(token) for token in nlp(comment)]
    print(token_list[max(0, token_list.index(searchword) - width) :
               min(len(token_list), token_list.index(searchword) + width + 1)])  
def concordance_string(comment, searchword, width):
    print(comment[ max(0, comment.find(searchword)-width) :
                  min(len(comment), comment.find(searchword)+len(searchword)+width)])

#This function prints words in their contexts across comments 
def print_concordances(searchword, selected_comments,
                       number_print = 10, width = 5,
                       words_or_characters = "character"):
    
    list_of_rows = [row for idx, row in selected_comments.iterrows()]
    random.shuffle(list_of_rows)
    
    if words_or_characters == "word":
        i = 0
        for row in list_of_rows:
            if searchword in row['body']:
                print('\033[1m' + 'Subreddit: ' + row['subreddit'] + '\033[0m')
                print('ID: ' + row['name'])
                concordance_list(row['body'], searchword.strip(), width)
                print(" ")
                i+=1
                if not i<number_print:
                    break           
    if words_or_characters == "character":
        i = 0
        for row in list_of_rows:
            if searchword in row['body']:
                print('\033[1m' + 'Subreddit: ' + row['subreddit'] + '\033[0m')
                print('ID: ' + row['name'])
                concordance_string(row['body'], searchword, width)
                print(" ")
                i+=1
                if not i<number_print:
                    break
                    
#This function calculates sentiment scores 
def add_sentiment_scores(selected_comments, model='asent'):
    #Get out all the sentiment scores
    if model == 'vader':
        sentiments = [sentMachine.polarity_scores(comment)['compound'] for comment in selected_comments['body']]
    elif model == 'asent':
        sentiments = [nlp(comment)._.polarity.compound for comment in selected_comments['body']]
    else:
        print('model has to be either "vader" or "asent"')
        return selected_comments
    #Add them to our dataset
    selected_comments["sentiment"] = sentiments
    return selected_comments

#This plots the sentiment scores for the whole subset
def plot_sentiment_together(comments_sentiment,
                           palette = "ocean", bins = 20):
    sns.displot(data=comments_sentiment, x="sentiment", 
                kde = True, palette = palette,
                bins = bins).set(xlim=(-1, 1))

#This plots the sentiment scores separately for each subreddit
def plot_sentiment_split(comments_sentiment,
                         bar_or_line = 'line',
                         palette = "ocean",
                         fill = True, rug = False):
    
    if bar_or_line == 'line':
        sns.displot(data=selected_comments, x="sentiment", 
                    kind = "kde", hue = "subreddit", 
                    rug = rug, palette = palette,
                    fill = fill, cut = 0).set(xlim=(-1, 1))
    
    elif bar_or_line == 'bar':
        sns.displot(data=selected_comments, x="sentiment",
                    hue = "subreddit", palette = palette,
                    stat = 'density', element = "step",
                    rug = rug, fill = fill).set(xlim=(-1, 1))
    else:
        print('you mispelled bar_or_line')
    
def print_by_sentiment(selected_comments,
                       number_print = 5,
                       low_or_high = "low",
                       only_print_ID = False):
    
    #Set the ascending/decending order
    if low_or_high == "low":
        ascending = True
    elif low_or_high == "high":
        ascending = False
    else:
        print('you mispelled low_or_high')
    #set counter
    i = 0
    
    #Go through each row in the dataset, ordered by sentiment score
    for idx, row in selected_comments.sort_values('sentiment', ascending = ascending).iterrows():
        #Print the contents
        print('\033[1m' + 'subreddit: ' + row['subreddit'] +'\033[0m')
        print('\033[1m' + 'ID: '+ str(row['name']) +'  sentiment: ' + str(row['sentiment']) +'\033[0m')
        if not only_print_ID:
            print(row['body'])
        print(" ")
        #Break when number_print reached
        i += 1
        if i>number_print:
            break

# visualize by comment id using asent
def visualize_sentiment(df, comment_id, style):
    temp = df[df['name']==comment_id]['body']
    doc = nlp(list(temp)[0])
    asent.visualize(doc, style)

    
#This function counts the different pos tags         
def count_pos_tags(selected_comments, types_to_count):
    if types_to_count=='ALL':
        types_to_count = ["ADJ", "INTJ", "NOUN", "PROPN", "VERB", "ADP", "AUX", "ADV",
                          "CONJ", "DET", "NUM", "PART", "PRON", "SCONJ", "PUNCT", "SYM"]
    wordtypecounter = collections.Counter()
    for idx, row in selected_comments.iterrows():
        for token in nlp(row['body']):
            if token.pos_ in types_to_count:
                wordtypecounter[token.pos_] += 1      
    return wordtypecounter

#This function plots the distribution of pos tags
def plot_wordtypes(wordtypecounter):
    # Pie chart, where the slices will be ordered and plotted counter-clockwise:
    labels = wordtypecounter.keys()
    sizes = wordtypecounter.values()
    
    cmap = plt.cm.Greens
    c = [*cmap(np.linspace(.6, .1, len(labels)))]

    fig1, ax1 = plt.subplots()
    ax1.pie(sizes, labels=labels, autopct='%1.1f%%',
            shadow=True, startangle=90, colors=c)
    ax1.axis('equal')
    plt.show()

    
# For looking at word types pr subreddit
def count_pos_tags_split(df, types_to_count, style):
    for subreddit in df['subreddit'].unique():
        tmp = df[df['subreddit']==subreddit]
        wordtypecounter_ = count_pos_tags(tmp, types_to_count)
        print(f'Subreddit: {subreddit}')
        if style == 'count':
            for wordtype, count in wordtypecounter_.most_common():
                print(f'{wordtype}: {count}')
            print('\n')
        if style == 'plot':
            plot_wordtypes(wordtypecounter_)

#This function counts named entities of some given types
def count_entities(selected_comments, included_entities):
    if included_entities == 'ALL':
        included_entities = ["CARDINAL", "DATE", "EVENT", "FAC", "GPE",
                             "LANGUAGE", "LAW", "LOC", "MONEY", "NORP",
                             "ORDINAL", "ORG", "PERCENT", "PERSON", "PRODUCT",
                             "QUANTITY", "TIME", "WORK_OF_ART"]
    entitycounter = collections.Counter()
    for idx, row in selected_comments.iterrows():
        for entity in nlp(row['body']).ents:
            if entity.label_ in included_entities:
                entitycounter[entity.text] +=1
    return entitycounter

    
#This function prints words in their contexts across comments 
def get_thread(id_to_check, number_of_steps = 10, print_comments_in_database = True):
    print('\033[1m' +'-- Comment ID being checked: ' + id_to_check + ' --' + '\033[0m')
    print('\033[1m' + "Printing following comments in the thead" + '\033[0m')
    new_id = id_to_check
    for i in range(number_of_steps):
        connection = sqlite3.connect('/kaggle/input/reddit-comments-may-2015/database.sqlite')
        cursor = connection.cursor()
        cursor.execute("SELECT body, name FROM May2015 WHERE parent_id = '{}'".format(new_id))
        output = cursor.fetchall()
        connection.close()
        if not output:
            print('\033[1m' + "That's all the following comments I could find in the database" + '\033[0m')
            break
        body = output[0][0]
        own_id = output[0][1]
        if print_comments_in_database:
            print('comment id: ' + own_id)
            print(body)
            print(' ')
        #Set own id as the next id to use for searching in parent id's
        new_id = own_id
    #First find the preceding comments
    print('\033[1m' + "Printing preceding comments (including this one) in the thread" + '\033[0m')
    new_id = id_to_check
    for i in range(number_of_steps):
        connection = sqlite3.connect('/kaggle/input/reddit-comments-may-2015/database.sqlite')
        cursor = connection.cursor()
        cursor.execute("SELECT body, parent_id, name FROM May2015 WHERE name = '{}'".format(new_id))
        output = cursor.fetchall()
        connection.close()
        if not output:
            print("Did not find the id (either the one you specified or another in the thread) - sorry!")
            break
        body = output[0][0]
        parent_id = output[0][1]
        own_id = output[0][2]
        if print_comments_in_database:
            print('comment id: ' + own_id)
            print(body)
            print(' ') 
        #If it's the top of the row
        if(parent_id.startswith('t3_')):
            print(parent_id)
            submission_id = parent_id.split('_')[1]
            print('\033[1m' + 'This is the top of the thread!' + '\033[0m')
            print('\033[1m' + 'Link to original submission: www.reddit.com/' + submission_id + '\033[0m')
            print('\033[1m' + 'Do not click it, but copy it into the browser' + '\033[0m')
            break
        #Set the parent id as the next id to check
        new_id = parent_id   
        
# This uses spacy to find linguistic patterns
def use_matcher(selected_comments, 
                matcher,
                keyword,
                wordtypes_to_return_dict = None, 
                lemmatize = False):
    #Set up empty wordcounter
    wordcounter = collections.Counter()
    # Set up list for ids
    ids = []
    #Go through each row
    for idx, row in selected_comments.iterrows():
        #Extract the comment text
        text = row['body']
        #NLP it
        doc = nlp(text)
        #Find all matches
        matches = matcher(doc)
        #Go through all matches
        for match_id, start, end in matches:
            #Get the string rep of the match
            string_id = nlp.vocab.strings[match_id]
            #Get the bit of the text that matched it
            span = doc[start:end]
            ids.append(row['name'])
            
            if wordtypes_to_return_dict:
                wordtypes_to_return = wordtypes_to_return_dict[string_id]
                #Go through each token in the matching text
                for token in span:
                    #Check if it is the kind of word we are looking for
                    if token.pos_ in wordtypes_to_return:
                        #Don't count word if it is the keyword
                        if not token.lower_==keyword:
                            #And count it
                            if lemmatize:
                                wordcounter[token.lemma_] += 1
                            else:
                                wordcounter[token.text] += 1
            else:
                if lemmatize:
                    span = ' '.join([token.lemma_ for token in span])
                wordcounter[str(span)] += 1
    return wordcounter, ids

def test_matcher(test_texts, 
                 matcher, 
                 keyword,
                 wordtypes_to_return_dict = None, 
                 lemmatize = False):
    #Set up empty wordcounter
    wordcounter = collections.Counter()
    #Go through each text
    for text in test_texts:
        #NLP it
        doc = nlp(text)
        #Find all matches
        matches = matcher(doc)
        #Go through all matches
        for match_id, start, end in matches:
            #Get the string rep of the match
            string_id = nlp.vocab.strings[match_id]
            #Get the bit of the text that matched it
            span = doc[start:end]
            
            if wordtypes_to_return_dict:
                wordtypes_to_return = wordtypes_to_return_dict[string_id]
                #Go through each token in the matching text
                for token in span:
                    #Check if it is the kind of word we are looking for
                    if token.pos_ in wordtypes_to_return:
                        #Don't count the keyword
                        if not token.lower_==keyword:
                            if lemmatize:
                                print(token.lemma_)
                            else:
                                print(token.text)
            else:
                if lemmatize:
                    span = ' '.join([token.lemma_ for token in span])
                print(span)

#This function gets out a dataframe with random comments for making idf scores
def get_random_comments(number_comments = 10000):
    con = sqlite3.connect("/kaggle/input/reddit-comments-may-2015/database.sqlite")
    # Load the data into a DataFrame
    out_df = pd.read_sql_query("SELECT body, subreddit FROM May2015 WHERE id IN (SELECT id FROM May2015 ORDER BY RANDOM() LIMIT {})".format(number_comments), con)
    con.close()
    return out_df


def tf_scores(df, characters_to_exclude) -> dict:
    '''
    get term frequecy scores from pandas dataframe
    '''
    lemma_list = []
    for text in selected_comments['body']:
        for token in nlp(text):
            if token.is_punct:
                continue
            if any([char in token.lemma_ for char in characters_to_exclude]):
                continue
            lemma_list.append(token.lemma_.lower())
    term_count = collections.Counter(lemma_list)
    return term_count


def idf_scores(selected_comment, random_comments) -> dict:
    """
    get idf-scores from two pandas df: one with the selected comments
    and one with random comments. 
    """
    docs = []
    for text in selected_comments['body']:
        docs.append([token.lemma_ for token in nlp(text)])
    for text in random_comments['body']:
        docs.append([token.lemma_ for token in nlp(text)])
    
    df = {}
    for doc in docs:
        for term in set(doc):
            term_ = term.lower()
            df[term_] = df.get(term_, 0) + 1
    n = len(df)
    idf_dict = {term: log10(n/freq) for term, freq in df.items()}
    return collections.Counter(idf_dict)


def tfidf_scores(tf: dict, idf: dict) -> dict:
    """
    get tf-idf scores from two dictionaries containing the tf and idf scores
    """        
    tfidf_dict = {term: log10(freq+1)*idf[term] for term, freq in tf.items()} 
    return collections.Counter(tfidf_dict)


def create_matcher(patterns: list, wordtypes_to_return: list):
    #This makes a new matcher machine :) 
    matcher = Matcher(nlp.vocab)
    wordtypes_to_return_dict = {}
    
    for i, (pattern, wordtypes) in enumerate(zip(patterns, wordtypes_to_return), start=1):
        name = f"pattern{i}"
        wordtypes_to_return_dict[name] = wordtypes
        matcher.add(name, [pattern])
    return matcher, wordtypes_to_return_dict

print('Done with setup')

## Selecting

First you must select the subreddits you want to analyze.

### Checking the amount of comments

We can see how many comments there are in specific subreddits, and how many times a keyword appears. If you don't want to search for a specific keyword, you can change the keyword to "ALL". Note that this runs much faster if you don't do keyword search.

In [ ]:
#We make a list with some subreddits we wanna check out.
#It can be longer than three if you want.
list_subreddits = ['AmericanPolitics',
                  'Ask_Politics',
                  'uspolitics']
# list_subreddits = ['ALL']

keyword = "Obama" # you can also write "ALL"

#Then we count how many comments there are in each.
count_comments(list_subreddits, keyword)

### Extract the comments you want to analyze [Mandatory]
Here we get out the specific comments we want to look. You can choose a number of subreddits by changing the list *list_subreddits* below. You can also change the keyword to a different keyword or to "ALL", if you want all of the comments in the subreddits you have chosen.

This needs to run before many other code chunks can run, because they use the extracted data here.

In [ ]:
#Select your subreddits
list_subreddits = ['AmericanPolitics',
                  'Ask_Politics',
                  'uspolitics']

#Put in a different keyword if you wanna select baed on that
keyword = "obama" # you can also write "ALL"

#This gets out the comments we want
selected_comments = get_specific_comments(list_subreddits, keyword, 
                                          number_comments = 1000)

#And this shows the first ten to us
selected_comments.head(10)

### View comments
Here you can view *n* random comments of the comments you have selected.

In [ ]:
# select number of comments you want to view
n = 5

comment_and_id = zip(list(selected_comments['body']), list(selected_comments['name']))

for i, (comment, comment_id) in enumerate(sample(list(comment_and_id), n), start=1):
    print(f'\033[1mComment {i}:\033[0m', f'ID = {comment_id}', comment, sep = '\n', end='\n\n')


View specific comment based on the ID

In [ ]:
# put in ID
comment_id = "t1_cqy6eqv"

# print comment
print(list(selected_comments[selected_comments['name']==comment_id]['body'])[0])

## Concordances: seeing how the word is used
This code shows examples of a keyword appearing

### Preparation

When you look at the context in which certain words appear we might not want to look at all the surrounding words. E.g. we might want to exclude stopwords or punctuation. The code below can be modified for doing so. 

The settings here I just made up. You can change this if you need to.

In [ ]:
#Choose a keyword to look for
keyword = " Iraq "

#This finds examples of words. Increase width to see more of the comments.
print_concordances(keyword, selected_comments,
                   number_print = 10, width = 20, words_or_characters = "character")

### Getting the full thread

You can look more closely at a specific comment by typing in the ID of the comment in the function below. It will then look for other comments that are answers to the comment and that came before the comment. Finally, it will provide a link (when possible) to the original reddit page.

In [ ]:
get_thread('t1_cria2uk')

## Counting and TF-IDF

TF-IDF (term frequency-inverse document frequency) can be calculated in multiple ways. You can read more [here](https://en.wikipedia.org/wiki/Tf%E2%80%93idf). The functions in this toolbox uses the following formula to calculate TF-IDF: 

$TFIDF = log_{10}(tf) \cdot log_{10}(N/df)$

Where $tf$ is the raw term-frequency counts, $df$ is the document frequency (i.e. how many documents the term appears in) and $N$ is the number of terms. In our example a document corresponds to a comment.

### Getting term frequency
First we count the number of times each word appear in the selected comments

In [ ]:
# characters that should not be counted - you can change this as you like
characters_to_exclude = ['\n', '/r', ' ', '|']

term_frequency = tf_scores(selected_comments, characters_to_exclude)
for word, count in term_frequency.most_common(100):
    print(f'{word}: {count}')

### Getting IDF scores

We now calculate the IDF scores.

$IDF = log_{10}(N/df)$

To calculate IDF scores, we need a bunch of random comments. We then calculate the idf scores using the random and the selected comments. Finally, we make sure that we do not save the random comments, as we don't need those anymore (and keeping them could lead to a memory error). Because of the random comments, this chunk will take a little bit longer to run.

In [ ]:
random_comments = get_random_comments(number_comments = 3000)
inverse_document_frequency = idf_scores(selected_comments, random_comments)
del random_comments

Here you can see the terms with the highest IDF scores. Included in this list are also terms from the random comments, thus, some of them might not appear in the comments of the subreddits you have chosen.

*NB: A high IDF scores means that the word is rare.*

In [ ]:
for word, idf in inverse_document_frequency.most_common(100):
    print(f'{word}: {idf}')

### Getting TF-IDF scores
From the TF and IDF scores we can calculate the final TF-IDF scores. You only get the TF-IDF scores for terms that appear in your selected comments.

In [ ]:
wordcounter_tfidf = tfidf_scores(term_frequency, inverse_document_frequency)
for word, tfidf in wordcounter_tfidf.most_common(100):
    print(f'{word}: {tfidf}')

Finally, this little code can show us the scores for some concrete words. Then we can compare a specific word's score between two subreddits, for example.

In [ ]:
#Here is where to put in the words to check
words_to_check = ["love", "car", "i", "war"]

#I didn't make a function of this so you can see how it looks. It's not so crazy.
#This goes through each word in the list above one by one
for word in words_to_check:
    #This gets out the score from the wordcounter
    score = wordcounter_tfidf[word]
    #And this prints the score next to the word
    print(f'{word}: {score}')

## Sentiment Analysis

The sentiment analysis is performed using the model [Asent](https://github.com/KennethEnevoldsen/asent). Asent is a dictionery-based sentiment analysis that is inspired by Vader. You can find more information about the model [here](https://kennethenevoldsen.github.io/asent/introduction.html).

### Trying it out
Here you can try out the sentiment analyser for specific texts. You can change the text below and see how that changes the sentiment. You could try adding negations (e.g. “not happy”), intensifiers (e.g. “very happy”) or contrastive conjugations (e.g. “but”).

In [ ]:
# here are two example texts - you can change them, to see how it performs on other text
pos_text = "I am very happy"
neg_text = "I am not happy"

In [ ]:
for text in [pos_text, neg_text]:
    doc = nlp(text)
    asent.visualize(doc, style="prediction")

In [ ]:
for text in [pos_text, neg_text]:
    doc = nlp(text)
    asent.visualize(doc, style="analysis")

In [ ]:
# compound scores
for text in [pos_text, neg_text]:
    doc = nlp(text)
    print(text)
    print('compound = ', doc._.polarity.compound, end='\n\n')

You can also find the score for specific words here

In [ ]:
# you can change the word to see the score for other words
word = 'happy'
doc = nlp(word)

for token in doc:
   print(token, "\t", token._.valence)

### Adding sentiment scores to your data
Here we add sentiment scores to the comments you extracted

In [ ]:
selected_comments = add_sentiment_scores(selected_comments, model='asent')

selected_comments.head(10)

### Plotting
Here we can plot the distribution of sentiments

In [ ]:
plot_sentiment_together(selected_comments,
                       bins = 20)

There are also these plots which show the different subreddits separately.

Try turning fill on and off and see what you like best.

You can find other color palettes [here](https://matplotlib.org/2.0.2/users/colormaps.html).

In [ ]:
plot_sentiment_split(selected_comments,
                    palette = "magma",
                    bar_or_line = 'bar',
                    fill = True)

### Printing the comments themselves

This function can print a few of the comments with either the highest or lowest sentiment. You can choose the number of comments you want to see be changing the parameter *number_print*, and you can change whether you want to see the comments with the highest or lowest sentiment scores by changing the parameter *low_or_high* to 'low'/'high'.

If you turn on *only_print_ID* then you will only get the ID numbers, which can be used with the get_thread function to get links to reddit and so on.
Why only ID's? Because some of the comments can be very long.

In [ ]:
print_by_sentiment(selected_comments,
                   number_print = 5,
                   low_or_high = "low",
                   only_print_ID = False)

You can use the comment IDs to visualize a specific comment and get a better understanding of the sentiment score.

In [ ]:
# change comment_id to the ID of the comment you want to visualize
comment_id = 't1_croa65v'
# you can visualize the comment using the style prediction or analysis
style = 'prediction'

visualize_sentiment(selected_comments, comment_id, style)


## Search based in linguistic analysis

This bit uses a software called SpaCy to do syntactic analysis!

There are many abbreviations. In the bootom of the toolbox, there is a list of all the abbreviations, and a function to see what they mean.

### Trying it out

First we can see what SpaCy returns when you ask it to look at syntactic structure

In [ ]:
text = "i sing terribly"

displacy.render(nlp(text), style="dep")

And we can see what it can do when it comes to figuring out types of entities

In [ ]:
text = """
Ohymgod, I went and shopped on Amazon this other day,
and I saw Steve Jobs' face, and the new poster with the Avengers.
I also bought a Star Wars shirt for Karen's brother
"""
displacy.render(nlp(text), style="ent")

### Counting wordtypes
We can make SpaCy count how often words of different types appear. 

In [ ]:
#Here you can select which wordtypes we are interested in
types_to_count = ['NOUN', 'ADJ', 'VERB', 'ADV']
#ADJ, INTJ, NOUN, PROPN, VERB, ADP, AUX,
#CONJ, DET, NUM, PART, PRON, SCONJ, PUNCT, SYM

# if you want all wordtype
# types_to_count = 'ALL'

#This counts all words of those types in your data
wordtypecounter = count_pos_tags(selected_comments, types_to_count)


In [ ]:
#And plots them
plot_wordtypes(wordtypecounter)

In [ ]:
# here you can see the exact count for the different wordtypes
for wordtype, count in wordtypecounter.most_common():
    print(f'{wordtype}: {count}')

In [ ]:
# If you have chosen more than one subreddit you can compare

# chose the word types
types_to_count = ['NOUN', 'ADJ', 'VERB']
# chose whether you want count or plot
style = 'count' # 'plot'

count_pos_tags_split(selected_comments, types_to_count, style)

### Counting Entities

We can also investigate what are the most common entities that appear in the selected comments.

In [ ]:
#Here we set which kinds of entities to include
included_entities = ["PERSON"]
#CARDINAL, DATE, EVENT, FAC, GPE,
#LANGUAGE, LAW, LOC, MONEY, NORP,
#ORDINAL, ORG, PERCENT, PERSON, PRODUCT,
#QUANTITY, TIME, WORK_OF_ART

# if you want all entities
# included_entities = 'ALL'

#We count how many there are of each entity
entitycounter = count_entities(selected_comments, included_entities)

for entity, count in entitycounter.most_common(50):
    print(f'{entity}: {count}')

### Looking for syntactic structures

In this bit, we can look for syntactic structures. You can read more [here](https://spacy.io/api/matcher).

First we create a list of patterns to look for

In [ ]:
#This keyword is used in the patterns below. Make sure to write it in all lowercase.
keyword = 'you'

# -- set up patterns to find -- #
#There are two patterns already made.
#Pattern 1 finds KEYWORD - be - ADJ. For example 'I am sad' or 'God is great'
#Pattern 2 KEYWORD - be - DET(optional) - amod(optional) - NOUN. For example 'I am the best hunter'.
patterns = [
    [{"LOWER": keyword}, {"LEMMA": 'be'}, {"POS": "ADJ"}],
    [{"LOWER": keyword}, {"LEMMA": 'be'}, {'POS': 'DET', 'OP': '?'}, {'DEP': 'amod', 'OP': '?'}, {"POS": "NOUN"}]
]

#This sets which wordtypes in the pattern should be counted. 
#Make sure the order is the same as for the patterns!
wordtypes_to_return = [
    ['NOUN', 'ADJ'],
    ['NOUN', 'ADV', 'ADJ']
]

matcher, wordtypes_to_return_dict = create_matcher(patterns, wordtypes_to_return)

Here you can try out how it works on some test text. You can modify the *test_text* if you want to.

You can both choose to have the function output the whole pattern that is matched or just the wordtypes from the list *wordtype_to_return*, which is defined in the chunk above. Try to comment/uncomment the line saying *wordtype_to_return_dict* and see how this changes the output.

Similarly, you can chose whether you want to lemmatize or not. You can change the argument *lemmatize* to True/False and see how this changes the output. What do you think would be the benefit of lemmatization?

In [ ]:
test_texts = ['you are cool but I also like cats',
             'dogs are cute',
             'you are the best friend!']

test_matcher(test_texts, matcher, keyword,
#              wordtypes_to_return_dict, 
             lemmatize = False)

Now we can find those patterns in all our data. The two arguments *wordtype_to_return_dict* and *lemmatize* works in the same way as they did for the test above. You can change these as you want. 

*Remember, if you want a different keyword, you can change it in the chunk above.*

In [ ]:
syntactic_wordcounter, comment_ids = use_matcher(selected_comments, matcher, keyword, 
#                                                  wordtypes_to_return_dict,
                                                 lemmatize = False)

In [ ]:
if not syntactic_wordcounter:
    print(f'The keyword "{keyword}" is not in the selected comments. Change your keyword.')
for word, count in syntactic_wordcounter.most_common(100):
    print(f'{word}: {count}')

In [ ]:
# look at the comment_ids for the specific patterns
m=0
for pattern, n in syntactic_wordcounter.items():
    n = n+m
    current_ids = comment_ids[m:n]
    for name in current_ids:
        print(f'{name}: {pattern}')
    m = n

Here is another pattern you could use

In [ ]:
# -- set up patterns to find -- #
# keyword = 'people' 
keyword = None  # if you don't wish to use a keyword put in None (without "")

if keyword:
    patterns = [
        [{"LOWER": 'as'}, {'POS': 'DET', 'OP': '?'}, {'DEP': 'amod', 'OP': '?'}, {"LOWER": keyword}]
    ]
else:
    patterns = [
        [{"LOWER": 'as'}, {'POS': 'DET', 'OP': '?'}, {'DEP': 'amod', 'OP': '?'}, {"POS": "NOUN"}]
    ]

#This sets which wordtypes in the pattern should be counted. 
#Make sure the order is the same as for the patterns!
wordtypes_to_return = [
    ['NOUN', 'ADV', 'ADJ']
]

matcher, wordtypes_to_return_dict = create_matcher(patterns, wordtypes_to_return)

In [ ]:
test_texts = [
    "as a doctor, I think...",
    "as concerned parents, we are convinced that..."
]

test_matcher(test_texts, matcher, keyword,
#              wordtypes_to_return_dict, 
             lemmatize = False)

In [ ]:
syntactic_wordcounter, comment_ids = use_matcher(selected_comments, matcher, keyword, 
#                                                  wordtypes_to_return_dict,
                                                 lemmatize = False)

In [ ]:
if not syntactic_wordcounter:
    print(f'The keyword "{keyword}" is not in the selected comments. Change your keyword.')
for word, count in syntactic_wordcounter.most_common(100):
    print(f'{word}: {count}')

In [ ]:
# look at the comment_ids for the specific patterns
m=0
for pattern, n in syntactic_wordcounter.items():
    n = n+m
    current_ids = comment_ids[m:n]
    for name in current_ids:
        print(f'{name}: {pattern}')
    m = n

In [ ]:
# put in ID
comment_id = "t1_cr7q1a3"

# print comment
print(list(selected_comments[selected_comments['name']==comment_id]['body'])[0])

Here is a list of abbreviations that SpaCy uses. Use the funciton to check what they mean.



In [ ]:
spacy.explain("amod")

# - Part-of-speech tag / word types - #
#AUX, ADJ, INTJ, NOUN, PROPN, VERB, ADP, AUX,
#CONJ, DET, NUM, PART, PRON, SCONJ, PUNCT, SYM

# - Entities - #
#CARDINAL, DATE, EVENT, FAC, GPE,
#LANGUAGE, LAW, LOC, MONEY, NORP,
#ORDINAL, ORG, PERCENT, PERSON, PRODUCT,
#QUANTITY, TIME, WORK_OF_ART

# - Dependencies - #
#ROOT, acl, acomp, advcl, advmod, agent, amod,
#appos, attr, aux, auxpass, case, cc, ccomp, 
#compound, conj, csubj, csubjpass, dative, 
#dep, det, dobj, expl, intj, mark, meta, neg,
#nmod, npadvmod, nsubj, nsubjpass, nummod,
#oprd, parataxis, pcomp, pobj, poss, preconj,
#predet, prep, prt, punct, quantmod, relcl, xcomp